# Notebook 2: Fine-Tuning (Transfer Learning — Phase 2)

## What changes
Instead of keeping the pretrained backbone frozen, we **unfreeze** the deeper layers
and train them with a **very small learning rate**.

## Why this helps
- Feature extraction uses generic ImageNet features → good but not optimal
- Fine-tuning **adapts** those features specifically to flowers
- Early layers (edges, textures) stay mostly the same
- Deeper layers (high-level patterns) shift toward flower-specific features

## Critical: Discriminative Learning Rates
- **Backbone** (pretrained layers): very small LR (1e-5) → gentle nudges
- **Head** (new layers): higher LR (1e-3) → learns faster

If you use the same high LR everywhere, you'll **destroy** the pretrained features!

In [ ]:
import sys
sys.path.insert(0, '..')

import torch
from src.data import get_dataloaders
from src.model import create_model, count_parameters, get_parameter_groups
from src.train import train_model
from src.evaluate import plot_training_history, get_predictions, plot_confusion_matrix

In [ ]:
DEVICE = (
    'mps' if torch.backends.mps.is_available()
    else 'cuda' if torch.cuda.is_available()
    else 'cpu'
)
print(f'Using device: {DEVICE}')

BATCH_SIZE = 32
NUM_EPOCHS = 20

In [ ]:
train_loader, val_loader, test_loader = get_dataloaders(
    data_dir='../data', batch_size=BATCH_SIZE
)

In [ ]:
# ---- Create Model (Fine-Tuning mode) ----
# Unfreeze from layer3 onward — deeper layers adapt to flowers
model = create_model(num_classes=102, mode='fine_tune', unfreeze_from='layer3')
model = model.to(DEVICE)

# Compare with Notebook 1 — many more trainable parameters now
count_parameters(model)

In [ ]:
# ---- Discriminative Learning Rates ----
# This is the KEY difference from naive fine-tuning
param_groups = get_parameter_groups(
    model,
    head_lr=1e-3,      # New head: learn fast
    backbone_lr=1e-5   # Pretrained layers: gentle nudges only
)

optimizer = torch.optim.AdamW(param_groups, weight_decay=1e-4)

# Cosine annealing: smoothly decreases LR → better convergence than step decay
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=NUM_EPOCHS, eta_min=1e-7
)

In [ ]:
# ---- Train ----
history = train_model(
    model, train_loader, val_loader, DEVICE,
    optimizer=optimizer, scheduler=scheduler,
    num_epochs=NUM_EPOCHS,
    save_path='../models/fine_tune_best.pth'
)

In [ ]:
plot_training_history(history, save_path='../results/fine_tune_curves.png')

In [ ]:
# ---- Test Evaluation ----
model.load_state_dict(torch.load('../models/fine_tune_best.pth', map_location=DEVICE))
preds, labels = get_predictions(model, test_loader, DEVICE)

test_acc = 100.0 * (preds == labels).mean()
print(f'\nFine-Tuned Test Accuracy: {test_acc:.1f}%')
print(f'Compare with Feature Extraction accuracy from Notebook 1!')

In [ ]:
plot_confusion_matrix(preds, labels, save_path='../results/fine_tune_confusion.png')

## Experiments to Try (for deeper understanding)

Change ONE thing at a time and observe the effect:

1. **Use the same LR for backbone and head** (e.g., 1e-3 for both)
   → Watch accuracy drop — you're destroying pretrained features

2. **Unfreeze from layer1 instead of layer3**
   → More parameters to train, but early features are already universal

3. **Remove data augmentation** (use eval transforms for training)
   → Overfitting will be much worse on this small dataset

4. **Try a larger model** (ResNet50 instead of ResNet18)
   → More capacity, but also more risk of overfitting

5. **Train from scratch** (no pretrained weights)
   → See how much worse it is — this proves transfer learning's value